In [ ]:
import math
import numpy as np
import pandas as pd
from collections import defaultdict
from matplotlib import pyplot as plt
import torch
from torchvision.transforms import Lambda
from torch.utils.data import Dataset, DataLoader
from torch import nn
from tqdm import tqdm
import time
import os

In [ ]:
class CTRDataset(Dataset):
    def __init__(self, data_path, feat_mapper=None, defaults=None,
                 min_threshold=4, num_feat=34,transform = None):
        self.NUM_FEATS, self.count, self.data = num_feat, 0, {}
        feat_cnts = defaultdict(lambda: defaultdict(int)) #???i??????embedding,??embedding????
        self.feat_mapper, self.defaults = feat_mapper, defaults
        self.field_dims = np.zeros(self.NUM_FEATS, dtype=np.int64)
        self.transform = transform
        with open(data_path) as f:
            for line in f:
                instance = {}
                values = line.rstrip('\n').split('\t')
                if len(values) != self.NUM_FEATS + 1:
                    continue
                label = np.float32([0, 0])
                label[int(values[0])] = 1
                instance['y'] = [np.float32(values[0])]
                for i in range(1, self.NUM_FEATS + 1):
                    feat_cnts[i][values[i]] += 1 #?i?feature???embedding???+1
                    instance.setdefault('x', []).append(values[i])
                self.data[self.count] = instance #{'y': [1.0], 'x': ['11417225884335159926',...,'631302449310544']}
                self.count = self.count + 1

        if self.feat_mapper is None and self.defaults is None:
            feat_mapper = {i: {feat for feat, c in cnt.items() if c >= min_threshold} for i, cnt in feat_cnts.items()}
            # ?feat_cnts?34?feature???embedding???feat_mapper,???????????min_threshold
            self.feat_mapper = {i: {feat_v: idx for idx, feat_v in enumerate(feat_values)}
                                for i, feat_values in feat_mapper.items()}
            # self.feat_mapper{key:1~34 ;value:DICT},DICT{key:?i?feature???embedding;value:??feat_mapper??i?dict??index ??:[0,len(feat_mapper[i])]}
            self.defaults = {i: len(feat_values) for i, feat_values in feat_mapper.items()}
            #???i?feature????embedding
        for i, fm in self.feat_mapper.items():
            self.field_dims[i - 1] = len(fm) +1 #????????
        self.offsets = np.array((0, *np.cumsum(self.field_dims)[:-1]))
        # ?? ???embedding?????? offsets[i]==len(self.feat_mapper[i-1])+1
        # ?i?feature:[offsets[i],offsets[i]+len(self.feat_mapper[i])-1]
        # ?i-1?feature????embedding:offsets[i-1]+len(self.feat_mapper[i-1])==offsets[i]-1
    def __len__(self):
        return self.count

    def __getitem__(self, idx):
        feat = np.array([self.feat_mapper[i + 1].get(v, self.defaults[i + 1]) #????????self.defaults[i + 1]
                         for i, v in enumerate(self.data[idx]['x'])])
        X = feat + self.offsets
        Y = self.data[idx]['y']
        if self.transform != None:
            X = self.transform(X)
            Y = self.transform(Y)
        return X,Y

In [ ]:
class FM(nn.Module):
    def __init__(self, field_dims, num_factors):
        super(FM, self).__init__()
        num_inputs = int(sum(field_dims)) #34???,??????????????
        self.embedding = nn.Embedding(num_inputs, num_factors)
        self.fc = nn.Embedding(num_inputs, 1)
        self.linear_layer = nn.Linear(len(field_dims),1,bias=True)

    def forward(self, x):
        # x.shape  [b,34]
        # self.embedding(x).shape [b,34, 20]
        # torch.sum(self.embedding(x), dim=1).shape [b,34]
        square_of_sum = torch.sum(self.embedding(x), dim=2) ** 2 #[b,34]
        sum_of_square = torch.sum(self.embedding(x) ** 2, dim=2) #[b,34]
        x = self.linear_layer(self.fc(x).sum(2)) + 0.5 * (square_of_sum - sum_of_square).sum(1,keepdims=True)
        x = torch.sigmoid(x)
        return x

In [ ]:
batch_size = 2048
data_dir = '../../data/ctr'
transform = Lambda(lambda x: torch.tensor(x))
train_data = CTRDataset(os.path.join(data_dir, 'train.csv'),transform=transform)
test_data = CTRDataset(os.path.join(data_dir, 'test.csv'),
                           feat_mapper=train_data.feat_mapper,
                           defaults=train_data.defaults,transform=transform)
train_iter = DataLoader(
    train_data, shuffle=True, batch_size=batch_size)
test_iter = DataLoader(
    test_data, shuffle=False,  batch_size=batch_size)


net = FM(train_data.field_dims,20)

# print(net(torch.tensor(train_data.__getitem__(0)[0])))
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)
loss = nn.BCELoss()
num_epochs = 30
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: 1/(epoch+1))

In [ ]:
def evaluate_accuracy(data_iter, net, device=None):
    """
    ??y?????
    :param data_iter:
    :param net:
    :param device:
    :return:
    """
    if device is None and isinstance(net, torch.nn.Module):
        # ?????device???net?device
        device = list(net.parameters())[0].device
    acc_sum, n = 0.0, 0
    with torch.no_grad():
        for X, y in data_iter:
            net.eval() # ????, ????dropout
            acc_sum += (net(X.to(device)) == y.to(device)).sum().cpu().item()
            net.train() # ??????
            n += y.shape[0]
    return acc_sum / n

def train(net, train_iter, test_iter, optimizer,loss, device, num_epochs,oneHotEncoder=None,scheduler = None):
    net = net.to(device)
    print("training on ", device)
    plt_epoch = []
    plt_train_acc = []
    plt_test_acc = []
    for epoch in range(num_epochs):
        plt_epoch.append(epoch)
        train_l_sum, train_acc_sum, n, batch_count, start = 0.0, 0.0, 0, 0, time.time()
        for X, y in tqdm(train_iter):
            X = X.to(device)
            y_oneHot = y
            if oneHotEncoder:
                y_oneHot = torch.tensor(oneHotEncoder.transform(y).toarray())
            y = y.to(device)
            y_oneHot = y_oneHot.to(device)
            y_hat = net(X)
            l = loss(y_hat, y_oneHot)
            optimizer.zero_grad()
            l.backward()
            # grad_clipping(net, 1)
            optimizer.step()
            train_l_sum += l.cpu().item()
            train_acc_sum += (y_hat == y).sum().cpu().item()
            # print((y_hat == y).sum().cpu().item(),y.shape[0])
            n += y.shape[0] # batch_num
            batch_count += 1
            # print('train_acc_sum:',train_acc_sum,'n',n)
        if scheduler:
            scheduler.step()
        test_acc = evaluate_accuracy(test_iter, net)
        print('epoch %d, loss %.4f, train acc %.3f, test acc %.3f, time %.1f sec'
              % (epoch + 1, train_l_sum / batch_count, train_acc_sum / n, test_acc, time.time() - start))
        plt_train_acc.append(train_acc_sum / n)
        plt_test_acc.append(test_acc)
    plt.plot(plt_epoch, plt_train_acc, color='r', label='train')  # r????
    plt.plot(plt_epoch, plt_test_acc, color='b', label='test')  # ????RGB?????
    plt.show()

In [ ]:
train(net, train_iter, test_iter, optimizer,loss, device, num_epochs,scheduler=scheduler)

In [ ]:
df = pd.read_table('../../data/ctr/train.csv',sep = '\t',header = None)
print(len(df[df[0]==1])/len(df))